In [16]:
print("Hello testing jupyter")

Hello testing jupyter


Global Sequence Alignment using the Needleman–Wunsch Algorithm

Implement a global alignment program for two biological sequences using dynamic programming. The program allows the user to choose between:
1. A simple scoring scheme (match, mismatch, gap), or
2. The PAM250 substitution matrix for protein sequence alignment.

Inputs:
* Sequence A (string)
* Sequence B (string)
* Scoring choice:
    * Simple Scoring:
        * Match score (integer)
        * Mismatch score (integer)
        * Gap penalty (integer)
    * Scoring with PAM250 matrix. Gap penalty still provided by user

Outputs:
1. Optimal alignment score
2. One optimal global alignment

How to run
1. Run all cells from top to bottom
2. Edit the input cell to provide sequences and scoring paramters
3. Execute the alignment cell to view the score and alignment

Notes
* Global alignment using Needleman-Wunsch
* Single gap penalty only

In [17]:
# Load dependencies 
import numpy as np
from typing import Dict, Tuple

In [18]:
# PAM250 Definition 
# Amino acid order used in PAM250
PAM250_AA_ORDER = list("ARNDCQEGHILKMFPSTWYV")

# PAM250 substitution matrix
# Rows and columns follow PAM250_AA_ORDER
PAM250 = np.array([
    [ 2,-2, 0, 0,-2, 0, 0, 1,-1,-1,-2,-1,-1,-4, 1, 1, 1,-6,-3, 0],
    [-2, 6, 0,-1,-4, 1,-1,-3, 2,-2,-3, 3, 0,-4, 0, 0,-1, 2,-4,-2],
    [ 0, 0, 2, 2,-4, 1, 1, 0, 2,-2,-3, 1,-2,-4, 0, 1, 0,-4,-2,-2],
    [ 0,-1, 2, 4,-5, 2, 3, 1, 1,-2,-4, 0,-3,-6,-1, 0, 0,-7,-4,-2],
    [-2,-4,-4,-5,12,-5,-5,-3,-3,-2,-6,-5,-5,-4,-3, 0,-2,-8, 0,-2],
    [ 0, 1, 1, 2,-5, 4, 2,-1, 3,-2,-2, 1,-1,-5, 0,-1,-1,-5,-4,-2],
    [ 0,-1, 1, 3,-5, 2, 4, 0, 1,-2,-3, 0,-2,-5,-1, 0, 0,-7,-4,-2],
    [ 1,-3, 0, 1,-3,-1, 0, 5,-2,-3,-4,-2,-3,-5, 0, 1, 0,-7,-5,-1],
    [-1, 2, 2, 1,-3, 3, 1,-2, 6,-2,-2, 0,-2,-2, 0,-1,-1,-3, 0,-2],
    [-1,-2,-2,-2,-2,-2,-2,-3,-2, 5, 2,-2, 2, 1,-2,-1, 0,-5,-1, 4],
    [-2,-3,-3,-4,-6,-2,-3,-4,-2, 2, 6,-3, 4, 2,-3,-3,-2,-2,-1, 2],
    [-1, 3, 1, 0,-5, 1, 0,-2, 0,-2,-3, 5, 0,-5,-1, 0, 0,-3,-4,-2],
    [-1, 0,-2,-3,-5,-1,-2,-3,-2, 2, 4, 0, 6, 0,-2,-2,-1,-4,-2, 2],
    [-4,-4,-4,-6,-4,-5,-5,-5,-2, 1, 2,-5, 0, 9,-5,-3,-3, 0, 7,-1],
    [ 1, 0, 0,-1,-3, 0,-1, 0, 0,-2,-3,-1,-2,-5, 6, 1, 0,-6,-5,-1],
    [ 1, 0, 1, 0, 0,-1, 0, 1,-1,-1,-3, 0,-2,-3, 1, 2, 1,-2,-3,-1],
    [ 1,-1, 0, 0,-2,-1, 0, 0,-1, 0,-2, 0,-1,-3, 0, 1, 3,-5,-3, 0],
    [-6, 2,-4,-7,-8,-5,-7,-7,-3,-5,-2,-3,-4, 0,-6,-2,-5,17, 0,-6],
    [-3,-4,-2,-4, 0,-4,-4,-5, 0,-1,-1,-4,-2, 7,-5,-3,-3, 0,10,-2],
    [ 0,-2,-2,-2,-2,-2,-2,-1,-2, 4, 2,-2, 2,-1,-1,-1, 0,-6,-2, 4]
], dtype=int)

# Map amino acid -> index
PAM250_INDEX: Dict[str, int] = {
    aa: i for i, aa in enumerate(PAM250_AA_ORDER)
}

def pam250_score(a: str, b: str) -> int:
    """Return PAM250 substitution score for amino acids a and b."""
    return PAM250[PAM250_INDEX[a], PAM250_INDEX[b]]

In [19]:
# Sanity check
assert pam250_score("A", "A") == 2
assert pam250_score("R", "R") == 6
assert pam250_score("N", "N") == 2
assert pam250_score("P", "H") == 0 

In [20]:
# Scoring 
def make_scoring_function(
    use_pam250: bool,
    match_score: int = 1,
    mismatch_score: int = -1
):
    """
    Returns a function score(a, b) that computes the substitution score
    for characters a and b.
    
    If use_pam250 is True, PAM250 is used.
    Otherwise, simple match and mismatch scoring is used.
    """
    
    if use_pam250:
        def score(a: str, b: str) -> int:
            return pam250_score(a, b)
    else:
        def score(a: str, b: str) -> int:
            return match_score if a == b else mismatch_score
    
    return score

In [26]:
# Testing scoring functionality. 

# Simple scoring
score_simple = make_scoring_function(
    use_pam250=False,
    match_score=2,
    mismatch_score=-1
)

print("Simple scoring A and A. Should be 2")
print(score_simple("A", "A"))  

print("Simple scoring A and G. Should be -1")
print(score_simple("A", "G"))  

# PAM250 scoring
score_pam = make_scoring_function(use_pam250=True)

print("PAM scoring A and A. Should be 2")
print(score_pam("A", "A"))  
print("PAM scoring W and W. Should be 17")
print(score_pam("W", "W"))  

Simple scoring A and A. Should be 2
2
Simple scoring A and G. Should be -1
-1
PAM scoring A and A. Should be 2
2
PAM scoring W and W. Should be 17
17


In [28]:
# Let's create two "matrices". 

# The DP score "matrix" F
# The traceback "matrix" TB

# Then, initiailze using a single gap penalty
def init_global_alignment_matrices(seq1: str, seq2: str, gap_penalty: int):
    """
    Initialize DP and traceback matrices for Needleman–Wunsch global alignment.

    Let:
      n = len(seq1)
      m = len(seq2)

    The DP score matrix F and traceback matrix TB both have shape (n+1) x (m+1).

    Interpretation:
      F[i, j] stores the optimal alignment score for the prefixes:
        seq1[0:i] and seq2[0:j]

    Example (n = 3, m = 3):

           j=0    j=1     j=2    j=3
        -----------------------------
    i=0 | F[0,0] F[0,1] F[0,2] F[0,3]
    i=1 | F[1,0] F[1,1] F[1,2] F[1,3]
    i=2 | F[2,0] F[2,1] F[2,2] F[2,3]
    i=3 | F[3,0] F[3,1] F[3,2] F[3,3]

    Initialization:
      - F[0,0] = 0
      - First column (j=0): cumulative gap penalties (seq1 aligned to gaps)
      - First row (i=0): cumulative gap penalties (seq2 aligned to gaps)

    Traceback directions stored in TB:
      'D' = diagonal (match/mismatch)
      'U' = up (gap in seq2)
      'L' = left (gap in seq1)
    """
    n = len(seq1)
    m = len(seq2)

    # Score matrix
    F = np.zeros((n + 1, m + 1), dtype=int)

    # Traceback matrix: store single-character direction codes
    TB = np.empty((n + 1, m + 1), dtype=object)
    TB[0, 0] = None

    # Initialize first column: seq2 is empty, so seq1 aligns to gaps
    for i in range(1, n + 1):
        F[i, 0] = F[i - 1, 0] + gap_penalty
        TB[i, 0] = "U"   # came from up (gap in seq2)

    # Initialize first row: seq1 is empty, so seq2 aligns to gaps
    for j in range(1, m + 1):
        F[0, j] = F[0, j - 1] + gap_penalty
        TB[0, j] = "L"   # came from left (gap in seq1)

    return F, TB

In [29]:
# Sanity check
seq1 = "ABC"
seq2 = "DEF"
gap_penalty = -2

F, TB = init_global_alignment_matrices(seq1, seq2, gap_penalty)

print("Score matrix F:")
print(F)

print("\nTraceback matrix TB:")
print(TB)

Score matrix F:
[[ 0 -2 -4 -6]
 [-2  0  0  0]
 [-4  0  0  0]
 [-6  0  0  0]]

Traceback matrix TB:
[[None 'L' 'L' 'L']
 ['U' None None None]
 ['U' None None None]
 ['U' None None None]]


In [30]:
# Fill in F and TB
def fill_global_alignment_matrices(
    seq1: str,
    seq2: str,
    F: np.ndarray,
    TB: np.ndarray,
    score_fn,
    gap_penalty: int
):
    """
    Fill DP score matrix F and traceback matrix TB
    using the Needleman–Wunsch recurrence.
    """
    n = len(seq1)
    m = len(seq2)

    for i in range(1, n + 1):
        for j in range(1, m + 1):

            diag = F[i - 1, j - 1] + score_fn(seq1[i - 1], seq2[j - 1])
            up   = F[i - 1, j] + gap_penalty
            left = F[i, j - 1] + gap_penalty

            # Choose the best move (tie-breaking: D > U > L)
            best_score = diag
            best_move = "D"

            if up > best_score:
                best_score = up
                best_move = "U"

            if left > best_score:
                best_score = left
                best_move = "L"

            F[i, j] = best_score
            TB[i, j] = best_move

In [33]:
# Testing with sample from the lecture to get global alignment score
seq1 = "ACTCG"
seq2 = "ACAGTAG"
gap_penalty = -1

score_fn = make_scoring_function(
    use_pam250=False,
    match_score=1,
    mismatch_score=0
)

F, TB = init_global_alignment_matrices(seq1, seq2, gap_penalty)
fill_global_alignment_matrices(seq1, seq2, F, TB, score_fn, gap_penalty)

print("Score matrix F:")
print(F)

print("\nTraceback matrix TB:")
print(TB)

Score matrix F:
[[ 0 -1 -2 -3 -4 -5 -6 -7]
 [-1  1  0 -1 -2 -3 -4 -5]
 [-2  0  2  1  0 -1 -2 -3]
 [-3 -1  1  2  1  1  0 -1]
 [-4 -2  0  1  2  1  1  0]
 [-5 -3 -1  0  2  2  1  2]]

Traceback matrix TB:
[[None 'L' 'L' 'L' 'L' 'L' 'L' 'L']
 ['U' 'D' 'L' 'D' 'L' 'L' 'D' 'L']
 ['U' 'U' 'D' 'L' 'L' 'L' 'L' 'L']
 ['U' 'U' 'U' 'D' 'D' 'D' 'L' 'L']
 ['U' 'U' 'D' 'D' 'D' 'D' 'D' 'D']
 ['U' 'U' 'U' 'D' 'D' 'D' 'D' 'D']]


In [34]:
# We have the score. Now we need the alignment
# Traceback reconstrution 
def traceback_global_alignment(seq1: str, seq2: str, TB: np.ndarray):
    """
    Reconstruct one optimal global alignment using the traceback matrix TB.

    Returns:
      aligned_seq1 (str)
      aligned_seq2 (str)
    """
    i = len(seq1)
    j = len(seq2)

    aligned_seq1 = []
    aligned_seq2 = []

    while i > 0 or j > 0:
        move = TB[i, j]

        if move == "D":
            aligned_seq1.append(seq1[i - 1])
            aligned_seq2.append(seq2[j - 1])
            i -= 1
            j -= 1

        elif move == "U":
            aligned_seq1.append(seq1[i - 1])
            aligned_seq2.append("-")
            i -= 1

        elif move == "L":
            aligned_seq1.append("-")
            aligned_seq2.append(seq2[j - 1])
            j -= 1

        else:
            raise ValueError(f"Invalid traceback direction at ({i}, {j})")

    # We built the alignment backwards
    aligned_seq1.reverse()
    aligned_seq2.reverse()

    return "".join(aligned_seq1), "".join(aligned_seq2)

In [35]:
# Testing with lecture example
aligned1, aligned2 = traceback_global_alignment(seq1, seq2, TB)

print("Optimal alignment score:", F[len(seq1), len(seq2)])
print(aligned1)
print(aligned2)

Optimal alignment score: 2
AC--TCG
ACAGTAG


In [37]:
# Example PAM250 alignment using Assignment 3.1 Task 2
seq1 = "PRKVV"
seq2 = "DPLVR"

gap_penalty = -2

# Create PAM250 scorer
score_fn = make_scoring_function(
    use_pam250=True
)

# Initialize and fill
F, TB = init_global_alignment_matrices(seq1, seq2, gap_penalty)
fill_global_alignment_matrices(seq1, seq2, F, TB, score_fn, gap_penalty)

# Traceback
aligned1, aligned2 = traceback_global_alignment(seq1, seq2, TB)

print("Optimal alignment score (PAM250):", F[len(seq1), len(seq2)])
print(aligned1)
print(aligned2)

Optimal alignment score (PAM250): 4
-PRKVV-
DP--LVR
